# 07 · Clean landlords: one landlord per building
**Inputs:** HPD registrations + contacts · **Output:** `data/processed/landlords_clean.parquet`

**The problem:** landlords hide behind a separate LLC per building, so "owner name" splits one landlord into hundreds.

**Our rule:** for each building's **latest** registration, pick ONE landlord key, in priority order:
1. **Managing-agent company** (normalized name), e.g. `FIRSTSERVICE RESIDENTIAL`.
2. **Managing-agent person + their office address**. A name alone is too common to trust.
3. **Owner's office address**: LLCs of one landlord usually share an office.

Buildings with the same key form one **portfolio** (`landlord_id`).

*Why not chain all the links?* We tested merging every building that shares any key with another (union-find / connected components). Chaining agent → office → agent → … glued **18,703 unrelated buildings** into one "landlord". The priority rule avoids that.

**Study window: January 1, 2025–October 31, 2026.** Annual comparisons should use January–October in both years. Housing prediction uses six-month histories and four-month outcomes; older figures and reports describe the previous study.

In [1]:
import sys, pathlib, json
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src import config, housing as H, clean_housing as C
from src.preprocess import CleaningReport

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_colwidth", 70)

def show_report(rep):
    return pd.DataFrame(rep.steps)[["step", "rows_before", "rows_after", "rows_removed", "detail"]]

def save(df, name, rep):
    df.to_parquet(C.OUT[name], index=False)
    (config.REPORTS_DIR / f"cleaning_report_{name}.json").write_text(json.dumps(rep.to_dict(), indent=2, default=str))
    print(f"Saved {len(df):,} rows × {df.shape[1]} columns → {C.OUT[name].relative_to(ROOT)}")

In [2]:
reg, con = H.load("hpd_registrations"), H.load("hpd_contacts")
rep = CleaningReport("landlords")
landlords = C.clean_landlords(reg, con, rep)
show_report(rep)

,step,rows_before,rows_after,rows_removed,detail
0,load_raw,187127,187127,0,"744,753 contacts"
1,latest_registration_per_bbl,187127,176715,10412,older registrations of the same building
2,build_portfolios,176715,173329,3386,"116,519 landlords"


In [3]:
print("Which key decided each building's landlord:")
display(landlords["landlord_key_type"].value_counts(dropna=False).to_frame("buildings"))
sizes = landlords.drop_duplicates("landlord_id")["portfolio_buildings"]
bands = pd.cut(sizes, [0, 1, 4, 9, 49, 99, 10_000], labels=["1", "2-4", "5-9", "10-49", "50-99", "100+"])
pd.DataFrame({"landlords": bands.value_counts().sort_index(),
              "buildings": landlords.drop_duplicates("landlord_id").groupby(bands, observed=False)["portfolio_buildings"].sum()})

Which key decided each building's landlord:


,buildings
landlord_key_type,
AGENT_P,69275
AGENT_CO,67989
OWNER_ADDR,36065
NaN,3386


,landlords,buildings
portfolio_buildings,,
1,108443,108443
2-4,8913,21458
5-9,1356,8696
10-49,1031,21004
50-99,103,7100
100+,59,10014


In [4]:
landlords.drop_duplicates("landlord_id").nlargest(15, "portfolio_buildings")[["landlord_key", "portfolio_buildings"]]

,landlord_key,portfolio_buildings
19376,AGENT_CO:FIRSTSERVICE RESIDENTIAL,522
57969,AGENT_CO:FREESTONE PROPERTY GROUP,434
9586,AGENT_CO:AKAM,357
15935,AGENT_CO:DOUGLAS ELLIMAN PROPERTY MANAGEMENT,301
19932,AGENT_CO:CONCORD MANAGEMENT OF NY,293
22626,AGENT_CO:REM RESIDENTIAL,262
23195,AGENT_CO:SHINDA MANAGEMENT,249
24464,AGENT_CO:WAVECREST MGMT GROUP,240
13130,AGENT_CO:ANDREWS ORGANIZATION,232
38741,AGENT_CO:NIEUW AMSTERDAM PROPERTY MANAGEMENT,226


In [5]:
# Sanity checks
assert landlords["bbl"].is_unique, "one row per building"
assert landlords["bbl"].str.fullmatch(r"[1-5]\d{9}").all()
print("Checks passed")
save(landlords, "landlords", rep)

Checks passed


Saved 176,715 rows × 9 columns → data\processed\landlords_clean.parquet
